# Titanic exp013 - Replace LogisticRegression with CatBoost on top of Title + FamilyGroup + SexPclass

この notebook は現在の baseline `Title + FamilyGroup + SexPclass` を維持したまま、モデルだけを `LogisticRegression` から `CatBoost` に置き換えて比較するためのものです。

仮説:
- `Title`、`FamilyGroup`、`SexPclass` を含む現在の特徴量でも、木ベースの boosted model の方が非線形や相互作用をうまく拾えるかもしれない
- 特徴量を増やさずモデルだけを変えることで、`exp007` より改善する余地があるかを確認する


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    COMP_DIR = cwd.parent
elif cwd.name == "titanic":
    COMP_DIR = cwd
else:
    COMP_DIR = Path("/home/sora/dev/kaggle/competitions/titanic")

DATA_DIR = COMP_DIR / "data"
SUBMISSION_DIR = COMP_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = SUBMISSION_DIR / "exp013_catboost.csv"

FEATURE_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Title", "FamilyGroup", "SexPclass"]
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked", "Title", "FamilyGroup", "SexPclass"]

TITLE_NORMALIZATION = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Dona": "Rare",
    "Dr": "Rare",
    "Rev": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Capt": "Rare",
    "Sir": "Rare",
    "Don": "Rare",
    "Jonkheer": "Rare",
    "Master": "Master",
    "Miss": "Miss",
    "Mr": "Mr",
    "Mrs": "Mrs",
}


In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    title = out["Name"].fillna("").astype(str).str.extract(r",\s*([^\.]+)\.", expand=False)
    out["Title"] = title.fillna("Rare").map(lambda x: TITLE_NORMALIZATION.get(x, "Rare")).astype(str)
    family_size = out["SibSp"].fillna(0) + out["Parch"].fillna(0) + 1
    out["FamilyGroup"] = pd.cut(
        family_size,
        bins=[0, 1, 4, 100],
        labels=["alone", "small", "large"],
        right=True,
    ).astype(str)
    out["Sex"] = out["Sex"].fillna("missing").astype(str)
    out["Embarked"] = out["Embarked"].fillna("missing").astype(str)
    out["Pclass"] = out["Pclass"].fillna(-1).astype(int).astype(str)
    out["SexPclass"] = out["Sex"] + "_" + out["Pclass"]
    return out


def build_model() -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False,
    )


In [ ]:
train_df = add_features(pd.read_csv(TRAIN_PATH))
test_df = add_features(pd.read_csv(TEST_PATH))

display(train_df[["Sex", "Pclass", "SexPclass", "Title", "FamilyGroup"]].head())
display(train_df.groupby("SexPclass", observed=False)["Survived"].agg(["mean", "count"]).sort_index())


In [ ]:
X = train_df[FEATURE_COLUMNS].copy()
y = train_df["Survived"].astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = build_model()
    model.fit(X_train, y_train, cat_features=CATEGORICAL_COLUMNS)
    valid_pred = model.predict(X_valid)
    score = accuracy_score(y_valid, valid_pred)
    scores.append(score)
    print(f"fold {fold}: {score:.5f}")

print("CV scores:", [round(score, 5) for score in scores])
print("CV mean:", round(float(pd.Series(scores).mean()), 5))
print("CV std:", round(float(pd.Series(scores).std(ddof=0)), 5))


比較メモ:

- `exp001` baseline CV mean: `0.79687`
- `exp006` Title + FamilyGroup CV mean: `0.82939`
- `exp007` Title + FamilyGroup + SexPclass CV mean: `0.83614`
- `exp013` exp007 features + CatBoost CV mean: `0.83839`


In [ ]:
final_model = build_model()
final_model.fit(X, y, cat_features=CATEGORICAL_COLUMNS)
test_predictions = final_model.predict(test_df[FEATURE_COLUMNS].copy()).astype(int)

submission_df = pd.DataFrame(
    {
        "PassengerId": test_df["PassengerId"],
        "Survived": test_predictions,
    }
)
submission_df.to_csv(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
display(submission_df.head())
